# Dependencies

In [1]:
# Install dependencies found in Pipfile
%pip install torch sentence-transformers qdrant-client psycopg2-binary tqdm pydantic python-dotenv sshtunnel

# for better output
%pip install --upgrade jupyter ipywidgets widgetsnbextension

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Configuration & SSH Tunnels

In [2]:
import time
import paramiko
from sshtunnel import SSHTunnelForwarder

# --- PATCH PARAMIKO ---
if not hasattr(paramiko, "DSSKey"):
    class DSSKey(paramiko.PKey):
        pass
    paramiko.DSSKey = DSSKey

# --- CONFIGURATION ---
SSH_HOST = "95.216.74.52"
SSH_USER = "lean"
SSH_PASSWORD = "l+VHTcsLfsdEiXlv" # Updated with the password you provided

# Postgres
PG_REMOTE_IP = "10.0.1.7"
PG_REMOTE_PORT = 5432
PG_LOCAL_PORT = 5433

# Qdrant (We will try both ports)
QDRANT_REMOTE_IP = "10.0.3.2"

# --- START TUNNEL ---
print("Establishing consolidated SSH Tunnel...")

tunnel = SSHTunnelForwarder(
    (SSH_HOST, 22),
    ssh_username=SSH_USER,
    ssh_password=SSH_PASSWORD,
    remote_bind_addresses=[
        (PG_REMOTE_IP, PG_REMOTE_PORT),      # Postgres
        (QDRANT_REMOTE_IP, 6333),             # Qdrant HTTP
        (QDRANT_REMOTE_IP, 6334)              # Qdrant GRPC
    ],
    local_bind_addresses=[
        ('127.0.0.1', PG_LOCAL_PORT),
        ('127.0.0.1', 6333),
        ('127.0.0.1', 6334)
    ],
    set_keepalive=10.0,
    mute_exceptions=True
)

tunnel.start()
time.sleep(2) # Wait for connections

if tunnel.is_active:
    print(f"✅ Tunnel Active!")
    print(f"   - Postgres:    localhost:{PG_LOCAL_PORT} -> {PG_REMOTE_IP}:{PG_REMOTE_PORT}")
    print(f"   - Qdrant HTTP: localhost:6333 -> {QDRANT_REMOTE_IP}:6333")
    print(f"   - Qdrant GRPC: localhost:6334 -> {QDRANT_REMOTE_IP}:6334")
else:
    print("❌ Tunnel failed to start.")

def close_tunnels():
    if tunnel.is_active:
        tunnel.stop()
    print("Tunnel closed.")

Establishing consolidated SSH Tunnel...
✅ Tunnel Active!
   - Postgres:    localhost:5433 -> 10.0.1.7:5432
   - Qdrant HTTP: localhost:6333 -> 10.0.3.2:6333
   - Qdrant GRPC: localhost:6334 -> 10.0.3.2:6334


# Database Class (db.py)

In [3]:
import psycopg2
from datetime import datetime
from typing import Dict, List, Optional

class Database:
    """PostgreSQL database operations."""
    
    def __init__(self, config: Dict):
        # Ensure we connect to localhost where the tunnel is bound
        self.conn = psycopg2.connect(**config)
        self.conn.set_session(readonly=True, autocommit=True)
    
    def get_datasets(self) -> List[Dict]:
        """Fetch all datasets with document counts."""
        with self.conn.cursor() as cur:
            cur.execute("""
                SELECT 
                    d.id, d.name, d.version, d.split, d.release_date, d.source_uri,
                    COUNT(did.document_id) as doc_count
                FROM dataset d
                LEFT JOIN document_in_dataset did ON d.id = did.dataset_id
                GROUP BY d.id, d.name, d.version, d.split, d.release_date, d.source_uri
                ORDER BY d.name
            """)
            columns = [desc[0] for desc in cur.description]
            return [dict(zip(columns, row)) for row in cur.fetchall()]
    
    def get_dataset_by_name(self, name: str) -> Optional[Dict]:
        """Get dataset by name."""
        return next((ds for ds in self.get_datasets() if ds['name'].lower() == name.lower()), None)
    
    def get_document_ids(self, dataset_id: str, limit: Optional[int] = None) -> List[str]:
        """Fetch document IDs for a dataset."""
        with self.conn.cursor() as cur:
            query = "SELECT document_id FROM document_in_dataset WHERE dataset_id = %s ORDER BY document_id"
            if limit:
                query += f" LIMIT {limit}"
            cur.execute(query, (dataset_id,))
            return [row[0] for row in cur.fetchall()]
    
    def get_documents(self, doc_ids: List[str]) -> List[Dict]:
        """Fetch full documents by IDs."""
        documents = []
        batch_size = 1000
        
        for i in range(0, len(doc_ids), batch_size):
            batch = doc_ids[i:i + batch_size]
            with self.conn.cursor() as cur:
                cur.execute("""
                    SELECT id, url, title, body, length, created_at, updated_at
                    FROM documents WHERE id = ANY(%s)
                """, (batch,))
                
                columns = [desc[0] for desc in cur.description]
                for row in cur.fetchall():
                    doc = dict(zip(columns, row))
                    # Convert datetime to ISO string
                    for k, v in doc.items():
                        if isinstance(v, datetime):
                            doc[k] = v.isoformat()
                    documents.append(doc)
        
        return documents
    
    def close(self):
        """Close connection."""
        self.conn.close()

# Ingestion Logic (ingest.py)

In [4]:
import sys
import os
import uuid
import requests
import grpc
from typing import List, Dict, Any, Set

import torch
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer

# --- PROGRESS BAR SETUP ---
# Try to use the widget-based tqdm, but fall back to text if it fails to render
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

# --- PIPELINE CONFIGURATION ---
DATASET_NAME = "small"
BATCH_SIZE = 100
PURGE_COLLECTION = False
RESUME_INGESTION = True

# DB Config
DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 5433,
    'database': 'postgres',
    'user': 'postgres',
    'password': '9IiVCS9s9ejH3QKO1aaFLyRzjAoTF87zZwp7cK0y0ofZoDvdq2uxF5OLm1hyAnRl',
    'sslmode': 'disable'
}

# Model Config
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

# --- CONNECTION CHECKER ---
def get_qdrant_client():
    print("🔎 Checking Qdrant connectivity...")
    
    # Check HTTP (6333)
    try:
        requests.get("http://127.0.0.1:6333/collections", timeout=2)
        print("✅ Detected Qdrant via HTTP (Port 6333)")
        return QdrantClient(
            host="127.0.0.1", 
            port=6333, 
            prefer_grpc=False, 
            https=False,
            api_key="DkJsZtSFdDjCClMz2vv5qAJmmrq2Si74"
        )
    except Exception:
        pass # Fail silently, try GRPC

    # Check GRPC (6334)
    try:
        with grpc.insecure_channel('127.0.0.1:6334') as channel:
            grpc.channel_ready_future(channel).result(timeout=2)
        print("✅ Detected Qdrant via GRPC (Port 6334)")
        return QdrantClient(
            host="127.0.0.1", 
            port=6334, 
            prefer_grpc=True, 
            https=False,
            api_key="DkJsZtSFdDjCClMz2vv5qAJmmrq2Si74"
        )
    except Exception:
        pass
        
    raise ConnectionError("Could not connect to Qdrant on port 6333 or 6334 via tunnel.")

# --- HELPER FUNCTIONS ---
def setup_qdrant(client: QdrantClient, vector_size: int, collection_name: str, purge: bool = False):
    if purge and client.collection_exists(collection_name):
        print(f"Purging existing collection '{collection_name}'...")
        client.delete_collection(collection_name=collection_name)

    if not client.collection_exists(collection_name):
        print(f"Creating collection '{collection_name}' with vector size {vector_size}...")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE)
        )
    else:
        print(f"Collection '{collection_name}' ready (existing).")

def chunk_text(text: str, tokenizer, chunk_size: int, overlap: int) -> List[str]:
    if not text: return []
    tokens = tokenizer.tokenize(text)
    if len(tokens) <= chunk_size: return [text]
    
    chunks = []
    stride = chunk_size - overlap
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    
    for i in range(0, len(token_ids), stride):
        chunk_ids = token_ids[i : i + chunk_size]
        chunks.append(tokenizer.decode(chunk_ids))
        if i + chunk_size >= len(token_ids): break
    return chunks

def get_existing_chunk_indices(qdrant: QdrantClient, collection_name: str, doc_id: Any) -> Set[int]:
    existing: Set[int] = set()
    try:
        scroll_res = qdrant.scroll(
            collection_name=collection_name,
            scroll_filter=models.Filter(must=[models.FieldCondition(key="doc_id", match=models.MatchValue(value=doc_id))]),
            limit=2048 
        )
        for point in scroll_res[0]:
            if point.payload and "chunk_index" in point.payload:
                existing.add(int(point.payload["chunk_index"]))
    except Exception:
        pass
    return existing

def process_batch(batch_docs, model, qdrant, dataset_name, collection_name, resume=True):
    total_chunks_new = 0
    total_chunks_skipped = 0
    tokenizer = model.tokenizer

    for doc in batch_docs:
        doc_id = doc['id']
        text = doc.get('body', '') or doc.get('title', '')
        if not text: continue

        chunks = chunk_text(text, tokenizer, CHUNK_SIZE, CHUNK_OVERLAP)
        if not chunks: continue

        existing_indices = get_existing_chunk_indices(qdrant, collection_name, doc_id) if resume else set()
        missing_indices = [i for i in range(len(chunks)) if i not in existing_indices]

        if not missing_indices:
            total_chunks_skipped += len(chunks)
            continue 

        missing_texts = [chunks[i] for i in missing_indices]
        embeddings = model.encode(missing_texts, batch_size=32, show_progress_bar=False, convert_to_numpy=True)

        points = []
        for local_pos, embedding in enumerate(embeddings):
            chunk_index = missing_indices[local_pos]
            point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{doc_id}_chunk_{chunk_index}"))
            meta = {
                "doc_id": doc_id, "chunk_index": chunk_index, "text": chunks[chunk_index],
                "dataset": dataset_name, "original_title": doc.get('title', ''), "url": doc.get('url', '')
            }
            points.append(models.PointStruct(id=point_id, vector=embedding.tolist(), payload=meta))

        qdrant.upsert(collection_name=collection_name, points=points)
        total_chunks_new += len(points)
        total_chunks_skipped += len(chunks) - len(points)

    return {"chunks_added": total_chunks_new, "chunks_skipped": total_chunks_skipped}

# --- MAIN EXECUTION ---
print(f"Initializing ingestion pipeline on device: {DEVICE}")

# 1. Connect to DB
try:
    db = Database(DB_CONFIG)
    print("Connected to PostgreSQL.")
except Exception as e:
    print(f"Failed to connect to DB: {e}")
    raise e

# 2. Connect to Qdrant
try:
    qdrant = get_qdrant_client()
except Exception as e:
    print(f"FATAL: {e}")
    db.close()
    raise e

# 3. Load Model
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}...")
model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE, trust_remote_code=True)
vector_size = model.get_sentence_embedding_dimension()

# 4. Select Dataset
dataset = db.get_dataset_by_name(DATASET_NAME)
if not dataset:
    print(f"Dataset '{DATASET_NAME}' not found.")
else:
    print(f"Processing dataset: {dataset['name']} ({dataset['doc_count']} docs)")
    
    # 5. Setup Collection
    setup_qdrant(qdrant, vector_size, DATASET_NAME, purge=PURGE_COLLECTION)

    # 6. Processing Loop
    doc_ids = db.get_document_ids(dataset['id'])
    print(f"Found {len(doc_ids)} documents. Starting ingestion...")

    cumulative_added = 0
    cumulative_skipped = 0

    # Force basic ASCII progress bar if widgets fail, or rich bar if available
    with tqdm(total=len(doc_ids), unit="docs", desc="Ingesting", mininterval=1.0) as pbar:
        for i in range(0, len(doc_ids), BATCH_SIZE):
            batch_ids = doc_ids[i: i + BATCH_SIZE]
            batch_docs = db.get_documents(batch_ids)
            stats = process_batch(batch_docs, model, qdrant, dataset['name'], DATASET_NAME, resume=RESUME_INGESTION)
            
            cumulative_added += stats["chunks_added"]
            cumulative_skipped += stats["chunks_skipped"]
            
            pbar.update(len(batch_ids))
            pbar.set_postfix(added=cumulative_added, skipped=cumulative_skipped)

    print("Ingestion complete.")
db.close()

Initializing ingestion pipeline on device: cpu
Connected to PostgreSQL.
🔎 Checking Qdrant connectivity...
✅ Detected Qdrant via HTTP (Port 6333)
Loading embedding model: Qwen/Qwen3-Embedding-0.6B...
Processing dataset: small (1612975 docs)
Collection 'small' ready (existing).
Found 1612975 documents. Starting ingestion...


Ingesting:   0%|          | 0/1612975 [00:00<?, ?docs/s]

KeyboardInterrupt: 

# Cleanup 

In [ ]:
close_tunnels()